# Aplicacao de iteradores em vendas

Este notebook mostra um caso pratico em que iteradores e geradores trabalham
juntos para processar dados de vendas armazenados em arquivo.

A classe `VendasIterator` le o arquivo linha por linha e converte cada registro
em uma tupla com categoria e valor. Em seguida, a funcao
`gerador_soma_vendas()` consome esse iterador e produz, para cada venda, o
total acumulado ate aquele momento.

Essa abordagem e util porque permite processar os dados de forma sequencial,
sem carregar todo o arquivo na memoria de uma vez, o que se torna importante
quando o volume de registros cresce.

In [1]:
# Iterador que percorre o arquivo de vendas uma linha por vez.
class VendasIterator:
    """Le um arquivo CSV simples e devolve categoria e valor por registro."""

    def __init__(self, arquivo):
        self.arquivo = arquivo
        self.file = open(arquivo, 'r', encoding='utf-8')

    def __iter__(self):
        return self

    def __next__(self):
        linha = self.file.readline()
        if not linha:
            # Fecha o arquivo quando nao houver mais dados para ler.
            self.file.close()
            raise StopIteration

        categoria, valor = linha.strip().split(',')
        return categoria, float(valor)


In [2]:
# Gerador que calcula o total acumulado de vendas durante a leitura.
def gerador_soma_vendas(arquivo):
    """Gera categoria, valor da venda e total acumulado."""

    total_vendas = 0
    for categoria, valor in VendasIterator(arquivo):
        total_vendas += valor
        yield categoria, valor, total_vendas

In [7]:
# Consome o gerador e, ao mesmo tempo, agrupa os valores por categoria.
contagem_vendas = {}
arquivo_vendas = '07-vendas.txt'

print('Categoria | Valor | Total Acumulado')

for categoria, valor, total in gerador_soma_vendas(arquivo_vendas):
    print(f"Categoria: {categoria}, Valor: {valor:.2f}, Total Acumulado: {total:.2f}")

    # Soma o valor vendido dentro de cada categoria.
    if categoria not in contagem_vendas:
        contagem_vendas[categoria] = 0
    contagem_vendas[categoria] += valor

for categoria, contagem in contagem_vendas.items():
    print(f'Total {categoria}: {contagem:.2f}')


# Resultado esperado:
# Categoria | Valor | Total Acumulado                    -> cabecalho para facilitar a leitura.
# Categoria: Eletronicos, Valor: 300.00, Total Acumulado: 300.00  -> primeira venda e primeiro acumulado geral.
# Categoria: Roupas, Valor: 150.00, Total Acumulado: 450.00       -> soma 150 ao acumulado anterior.
# Categoria: Eletronicos, Valor: 450.00, Total Acumulado: 900.00  -> nova venda de eletronicos acumulada no total geral.
# Categoria: Alimentos, Valor: 100.00, Total Acumulado: 1000.00   -> o acumulado geral chega a 1000.00.
# Categoria: Roupas, Valor: 200.00, Total Acumulado: 1200.00      -> roupas passa a somar 350.00 no total da categoria.
# Categoria: Eletronicos, Valor: 250.00, Total Acumulado: 1450.00 -> ultima venda processada.
# Total Eletronicos: 1000.00 -> soma final da categoria Eletronicos.
# Total Roupas: 350.00       -> soma final da categoria Roupas.
# Total Alimentos: 100.00    -> soma final da categoria Alimentos.


Categoria | Valor | Total Acumulado
Categoria: Eletrônicos, Valor: 300.00, Total Acumulado: 300.00
Categoria: Roupas, Valor: 150.00, Total Acumulado: 450.00
Categoria: Eletrônicos, Valor: 450.00, Total Acumulado: 900.00
Categoria: Alimentos, Valor: 100.00, Total Acumulado: 1000.00
Categoria: Roupas, Valor: 200.00, Total Acumulado: 1200.00
Categoria: Eletrônicos, Valor: 250.00, Total Acumulado: 1450.00
Total Eletrônicos: 1000.00
Total Roupas: 350.00
Total Alimentos: 100.00
